# Libs & Setup

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import logging
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta
import math
from torch import optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.optim import AdamW
from utils import inspect, inverse_transform, monte_carlo_statistic, calc_expected_returns, calc_covariance, calc_volatility
from utils.paths import CHECKPOINTS_DIR, REPORTS_SIM_DIR, RESULTS_DIR, REPORTS_QS_DIR
from pypfopt import risk_models, expected_returns, plotting, EfficientFrontier

import optuna
from optuna.trial import TrialState
import os
import random

# Own Libs
from config import *
from entities import *
from strategies import *
from datasets import *
from engine import Engine
from models import DiffusionTransformer, Diffusion

/home/narodom.y@FUSION.LAB/.conda/envs/tsgenai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

## Random Seed

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # สำหรับ GPU

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(78)

In [3]:
# cfg = TrainConfig(epochs=2, window_size=64, device=torch.device("cuda:1"))


device = torch.device("cuda:0") # if torch.cuda.is_available() else "cpu")
batch_size = 32
batch_size_exp = 1
epochs = 1000

# Window
window_size = 64
stride = 1

# Simulation
steps_sim = 90
paths_sim = 1000

# Optimizer
lr: float = 1e-4
weight_decay: float = 1e-6
betas: tuple = (0.9, 0.999)
eps: float = 1e-8

# Optuna
n_trials = 100
min_resource = 3
max_resource = 50
reduction_factor = 3
EPOCHS_PER_TRIAL = 50


# Model
ddpm = {
    'timesteps': int(1000),
    'beta_start': 0.0001,
    'beta_end': 0.02
}

ddpm_transformer = {
    'window_size': window_size,
    'd_model': 64,
    'nhead': 4,
    'num_layers': 32,
    'dim_feedforward': 512,
    'dropout': 0.1
}


# Data
time_range = {
    'start_date': '2021-01-01',
    'end_date': '2024-12-31'
}

# Indicator & condition
time_prd = 20
day_shift = 1

# N channels
n_prices = 1
n_targets = 1
n_conditions = 3


ckpt_name = f"ddpm_transformer_d{ddpm_transformer['d_model']}_l{ddpm_transformer['num_layers']}"
checkpoint_dir = os.path.join(RESULTS_DIR,"experimental_checkpoints_norm_steps_day120")
checkpoint_filename = f"tr{n_trials}_d{ddpm_transformer['d_model']}_dff{ddpm_transformer['dim_feedforward']}_l{ddpm_transformer['num_layers']}_h{ddpm_transformer['nhead']}_t{ddpm['timesteps']}.pt"

In [5]:
def time_range_info(df):
    info = (df.index.min(), df.index.max())
    print(f"Data range: {info[0]} to {info[1]}")

    duration = df.index.max() - df.index.min()
    print(f"Total duration: {duration}")

def time_range_mask(df, start_date, end_date):
    mask = (df.index >= start_date) & (df.index <= end_date)
    return mask

# Data

In [4]:
symbols = ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO']
freq = "1d"

# Basket
basket = Basket(symbols=symbols)
basket.load_all_assets(freq=freq)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

Basket data shape: (2760, 70)


AAPL                                                \
                Close       High        Low       Open       Volume   
Date                                                                  
2015-01-02  24.261047  24.729270  23.821672  24.718174  212818400.0   
2015-01-05  23.577574  24.110150  23.391173  24.030263  257142000.0   
2015-01-06  23.579794  23.839424  23.218085  23.641928  263188400.0   
2015-01-07  23.910433  24.010290  23.677430  23.788384  160423600.0   
2015-01-08  24.829119  24.886815  24.121236  24.238848  237458000.0   

                 TSLA                                             ...   AMD  \
                Close       High        Low       Open    Volume  ... Close   
Date                                                              ...         
2015-01-02  14.620667  14.883333  14.217333  14.858000  71466000  ...  2.67   
2015-01-05  14.006000  14.433333  13.810667  14.303333  80527500  ...  2.66   
2015-01-06  14.085333  14.280000  13.614000  14.004000  93928500  ...  2.63   
2015-01-07  14.063333  14.318667  13.985333  14.223333  44526000  ...  2.58   
2015-01-08  14.041333  14.253333  14.000667  14.187333  51637500  ...  2.61   

                                               CSCO                        \
            High   Low  Open      Volume      Close       High        Low   
Date                                                                        
2015-01-02  2.67  2.67  2.67         0.0  19.815605  20.181631  19.650534   
2015-01-05  2.70  2.64  2.67   8878200.0  19.420874  19.700776  19.377812   
2015-01-06  2.66  2.55  2.65  13912500.0  19.413698  19.865848  19.406522   
2015-01-07  2.65  2.54  2.63  12377600.0  19.593126  19.664896  19.363463   
2015-01-08  2.65  2.56  2.59  11136600.0  19.743843  20.160107  19.715135   

                                   
                 Open      Volume  
Date                               
2015-01-02  19.995029  22926500.0  
2015-01-05  19.607475  29460600.0  
2015-01-06  19.478291  47297600.0  
2015-01-07  19.478295  27570800.0  
2015-01-08  19.765374  40907000.0  

[5 rows x 70 columns]

In [7]:
time_range_info(basket.data)

for symbol, asset in basket.assets.items():
    mask = time_range_mask(asset.data, time_range['start_date'], time_range['end_date'])
    asset.data = asset.data[mask]

time_range_info(basket.data)

Data range: 2015-01-02 00:00:00 to 2025-12-22 00:00:00
Total duration: 4007 days 00:00:00
Data range: 2021-01-04 00:00:00 to 2024-12-31 00:00:00
Total duration: 1457 days 00:00:00


In [8]:
targets = ["Close"]
features = basket.get_unique_features()
print(f"Features:\t{features}\nTargets:\t{targets}")

Features:	['Close', 'High', 'Low', 'Open', 'Volume']
Targets:	['Close']


In [9]:
df = basket.data
df.ffill(inplace=True)
df.head()

AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-01-04  126.096588  130.189048  123.514437  130.101356  143301900   
2021-01-05  127.655609  128.366929  125.141666  125.589894   97664900   
2021-01-06  123.358521  127.694587  123.144152  124.449847  155088000   
2021-01-07  127.567940  128.259768  124.586290  125.073488  109578200   
2021-01-08  128.668976  129.234127  126.895568  129.039236  105158200   

                  TSLA                                                 ...  \
                 Close        High         Low        Open     Volume  ...   
Date                                                                   ...   
2021-01-04  243.256668  248.163330  239.063339  239.820007  145914600  ...   
2021-01-05  245.036667  246.946671  239.733337  241.220001   96735600  ...   
2021-01-06  251.993332  258.000000  249.699997  252.830002  134100000  ...   
2021-01-07  272.013336  272.329987  258.399994  259.209991  154496700  ...   
2021-01-08  293.339996  294.829987  279.463318  285.333344  225166500  ...   

                  AMD                                                  CSCO  \
                Close       High        Low       Open    Volume      Close   
Date                                                                          
2021-01-04  92.300003  96.059998  90.919998  92.110001  51802600  38.239326   
2021-01-05  92.769997  93.209999  91.410004  92.099998  34208000  38.256729   
2021-01-06  90.330002  92.279999  89.459999  91.620003  51911700  38.622074   
2021-01-07  95.160004  95.510002  91.199997  91.330002  42897200  39.109196   
2021-01-08  94.580002  96.400002  93.269997  95.980003  39816400  39.196186   

                                                       
                 High        Low       Open    Volume  
Date                                                   
2021-01-04  38.595972  37.708707  38.543782  24392500  
2021-01-05  38.335017  37.734811  37.995770  17763700  
2021-01-06  39.030909  38.178440  38.387210  21823100  
2021-01-07  39.239677  38.422000  38.448099  18218800  
2021-01-08  39.500638  38.491593  38.691662  20936300  

[5 rows x 70 columns]

In [10]:
assets = df.columns.get_level_values(0).unique()
assets

Index(['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO',
       'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO'],
      dtype='object')

In [11]:
processed_dfs = []


for symbol in assets:
    asset_df = df.xs(symbol, level=0, axis=1).copy()
    for target in targets:
        s = asset_df[target]
        close_price = asset_df['Close']
        volume = asset_df['Volume']
        high = asset_df['High']
        low = asset_df['Low']
        # Calc Log Return
        asset_df[f"Log_Returns {target}"] = np.log(s).diff()

        sma_volume = ta.SMA(volume, timeperiod=time_prd)
        # asset_df[f"SMA_Volume"] = sma_volume
        # asset_df[f"SMA_{time_prd} {target}"] = ta.SMA(s, timeperiod=time_prd)

        asset_df[f"Norm_Volume"] = ((volume - sma_volume) / sma_volume).shift(day_shift)
        asset_df[f"Distance_SMA_{time_prd} {target}"] = ((close_price - ta.SMA(s, timeperiod=time_prd)) / close_price).shift(day_shift)
        # asset_df[f"Distance_EMA_{time_prd} {target}"] = ((close_price - ta.EMA(s, timeperiod=time_prd)) / close_price).shift(day_shift)

        typical_price = (high + low + close_price) / 3
        tp_vol = typical_price * volume
        # asset_df[f"VWAP"] = tp_vol.cumsum() / volume.cumsum()
        vwap = tp_vol.cumsum() / volume.cumsum()
        # asset_df[f"VWAP"] = vwap
        asset_df[f"Norm_VWAP"] = np.log(vwap).diff().shift(1)


        # asset_df[f"SMA_{time_prd} {target}"] = ta.SMA(s, timeperiod=time_prd).shift(day_shift)
        # asset_df[f"EMA_{time_prd} {target}"] = ta.EMA(s, timeperiod=time_prd).shift(day_shift)
        # asset_df[f"RSI_{time_prd} {target}"] = ta.RSI(s, timeperiod=time_prd).shift(day_shift)
    asset_df.columns = pd.MultiIndex.from_product([[symbol], asset_df.columns])
    processed_dfs.append(asset_df)

df_updated = pd.concat(processed_dfs, axis=1)
df_updated.head()

AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-01-04  126.096588  130.189048  123.514437  130.101356  143301900   
2021-01-05  127.655609  128.366929  125.141666  125.589894   97664900   
2021-01-06  123.358521  127.694587  123.144152  124.449847  155088000   
2021-01-07  127.567940  128.259768  124.586290  125.073488  109578200   
2021-01-08  128.668976  129.234127  126.895568  129.039236  105158200   

                                                                          \
           Log_Returns Close Norm_Volume Distance_SMA_20 Close Norm_VWAP   
Date                                                                       
2021-01-04               NaN         NaN                   NaN       NaN   
2021-01-05          0.012288         NaN                   NaN       NaN   
2021-01-06         -0.034241         NaN                   NaN  0.001455   
2021-01-07          0.033554         NaN                   NaN -0.006358   
2021-01-08          0.008594         NaN                   NaN  0.001416   

                  TSLA  ...       AMD       CSCO                        \
                 Close  ... Norm_VWAP      Close       High        Low   
Date                    ...                                              
2021-01-04  243.256668  ...       NaN  38.239326  38.595972  37.708707   
2021-01-05  245.036667  ...       NaN  38.256729  38.335017  37.734811   
2021-01-06  251.993332  ... -0.002695  38.622074  39.030909  38.178440   
2021-01-07  272.013336  ... -0.008766  39.109196  39.239677  38.422000   
2021-01-08  293.339996  ...  0.004948  39.196186  39.500638  38.491593   

                                                               \
                 Open    Volume Log_Returns Close Norm_Volume   
Date                                                            
2021-01-04  38.543782  24392500               NaN         NaN   
2021-01-05  37.995770  17763700          0.000455         NaN   
2021-01-06  38.387210  21823100          0.009505         NaN   
2021-01-07  38.448099  18218800          0.012534         NaN   
2021-01-08  38.691662  20936300          0.002222         NaN   

                                            
           Distance_SMA_20 Close Norm_VWAP  
Date                                        
2021-01-04                   NaN       NaN  
2021-01-05                   NaN       NaN  
2021-01-06                   NaN -0.000800  
2021-01-07                   NaN  0.004101  
2021-01-08                   NaN  0.003558  

[5 rows x 126 columns]

In [12]:
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])

AAPL   Log_Returns Close         1
       Norm_Volume              20
       Distance_SMA_20 Close    20
       Norm_VWAP                 2
TSLA   Log_Returns Close         1
       Norm_Volume              20
       Distance_SMA_20 Close    20
       Norm_VWAP                 2
MSFT   Log_Returns Close         1
       Norm_Volume              20
       Distance_SMA_20 Close    20
       Norm_VWAP                 2
NVDA   Log_Returns Close         1
       Norm_Volume              20
       Distance_SMA_20 Close    20
       Norm_VWAP                 2
GOOGL  Log_Returns Close         1
       Norm_Volume              20
       Distance_SMA_20 Close    20
       Norm_VWAP                 2
AMZN   Log_Returns Close         1
       Norm_Volume              20
       Distance_SMA_20 Close    20
       Norm_VWAP                 2
GOOG   Log_Returns Close         1
       Norm_Volume              20
       Distance_SMA_20 Close    20
       Norm_VWAP                 2
META   Log_Returns C

In [13]:
df_updated.dropna(inplace=True)
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])
df_updated

Series([], dtype: int64)


AAPL                                                \
                 Close        High         Low        Open    Volume   
Date                                                                   
2021-02-02  131.533722  132.819917  131.163447  132.254765  83305400   
2021-02-03  130.510605  132.293751  130.189052  132.283998  89880900   
2021-02-04  133.872253  133.881992  131.143942  132.810165  84183100   
2021-02-05  133.457504  134.101570  132.579243  134.033268  75693800   
2021-02-08  133.603882  133.652677  131.661931  132.745127  71297200   
...                ...         ...         ...         ...       ...   
2024-12-24  257.286682  257.296626  254.386957  254.586262  23234700   
2024-12-26  258.103729  259.179926  256.718662  257.276679  27237100   
2024-12-27  254.685867  257.784882  252.164818  256.917934  42355300   
2024-12-30  251.307877  252.603281  249.863009  251.337769  35557500   
2024-12-31  249.534180  252.384064  248.547676  251.547039  39480700   

                                                                          \
           Log_Returns Close Norm_Volume Distance_SMA_20 Close Norm_VWAP   
Date                                                                       
2021-02-02          0.006317   -0.094484              0.007813  0.000022   
2021-02-03         -0.007809   -0.271330              0.011994  0.000484   
2021-02-04          0.025432   -0.211129              0.003154  0.000255   
2021-02-05         -0.003103   -0.237409              0.024259  0.000712   
2021-02-08          0.001096   -0.303623              0.019020  0.000691   
...                      ...         ...                   ...       ...   
2024-12-24          0.011413   -0.189446              0.040126  0.000321   
2024-12-26          0.003171   -0.506302              0.046114  0.000187   
2024-12-27         -0.013331   -0.409495              0.044508  0.000223   
2024-12-30         -0.013352   -0.090463              0.027644  0.000335   
2024-12-31         -0.007083   -0.242196              0.011626  0.000270   

                  TSLA  ...       AMD       CSCO                        \
                 Close  ... Norm_VWAP      Close       High        Low   
Date                    ...                                              
2021-02-02  290.929993  ... -0.002271  39.865974  39.961657  39.518027   
2021-02-03  284.896667  ... -0.001136  39.813793  40.153041  39.613724   
2021-02-04  283.329987  ... -0.001061  41.101185  41.162075  39.813783   
2021-02-05  284.076660  ... -0.001076  41.823177  42.049341  41.318653   
2021-02-08  287.806671  ... -0.001037  42.571262  42.919210  42.240716   
...                ...  ...       ...        ...        ...        ...   
2024-12-24  462.279999  ...  0.000079  58.345390  58.345390  57.321788   
2024-12-26  454.130005  ...  0.000049  58.472118  58.550109  57.906701   
2024-12-27  431.660004  ...  0.000048  58.111423  58.511116  57.653238   
2024-12-30  417.410004  ...  0.000058  57.701981  57.896953  56.941591   
2024-12-31  403.839996  ...  0.000047  57.711735  57.887210  57.292545   

                                                               \
                 Open    Volume Log_Returns Close Norm_Volume   
Date                                                            
2021-02-02  39.596315  16520900          0.009427   -0.196633   
2021-02-03  39.796395  13173600         -0.001310   -0.193696   
2021-02-04  39.900768  22285700          0.031824   -0.349778   
2021-02-05  41.379544  25488600          0.017414    0.098723   
2021-02-08  42.240716  25215400          0.017729    0.234508   
...               ...       ...               ...         ...   
2024-12-24  57.321788   9922300          0.014643   -0.161050   
2024-12-26  58.121168   8524500          0.002170   -0.507117   
2024-12-27  58.072428  13021400         -0.006188   -0.570487   
2024-12-30  57.604496  12948100         -0.007071   -0.344432   
2024-12-31  57.731232  14173000          0.000169   -0.350297 

In [14]:
drop_cols = ['Open', 'High', 'Low', 'Volume']

df_final = df_updated.drop(columns=drop_cols, level=1).copy()
df_final

AAPL                                                      \
                 Close Log_Returns Close Norm_Volume Distance_SMA_20 Close   
Date                                                                         
2021-02-02  131.533722          0.006317   -0.094484              0.007813   
2021-02-03  130.510605         -0.007809   -0.271330              0.011994   
2021-02-04  133.872253          0.025432   -0.211129              0.003154   
2021-02-05  133.457504         -0.003103   -0.237409              0.024259   
2021-02-08  133.603882          0.001096   -0.303623              0.019020   
...                ...               ...         ...                   ...   
2024-12-24  257.286682          0.011413   -0.189446              0.040126   
2024-12-26  258.103729          0.003171   -0.506302              0.046114   
2024-12-27  254.685867         -0.013331   -0.409495              0.044508   
2024-12-30  251.307877         -0.013352   -0.090463              0.027644   
2024-12-31  249.534180         -0.007083   -0.242196              0.011626   

                            TSLA                                \
           Norm_VWAP       Close Log_Returns Close Norm_Volume   
Date                                                             
2021-02-02  0.000022  290.929993          0.038519   -0.305379   
2021-02-03  0.000484  284.896667         -0.020956   -0.311081   
2021-02-04  0.000255  283.329987         -0.005514   -0.470524   
2021-02-05  0.000712  284.076660          0.002632   -0.523718   
2021-02-08  0.000691  287.806671          0.013045   -0.411586   
...              ...         ...               ...         ...   
2024-12-24  0.000321  462.279999          0.070991   -0.190564   
2024-12-26  0.000187  454.130005         -0.017787   -0.323247   
2024-12-27  0.000223  431.660004         -0.050745   -0.139047   
2024-12-30  0.000335  417.410004         -0.033569   -0.080851   
2024-12-31  0.000270  403.839996         -0.033050   -0.288918   

                                            ...         AMD                    \
           Distance_SMA_20 Close Norm_VWAP  ...       Close Log_Returns Close   
Date                                        ...                                 
2021-02-02              0.012481  0.000017  ...   88.860001          0.013596   
2021-02-03              0.041603  0.001551  ...   87.889999         -0.010976   
2021-02-04              0.014311  0.001010  ...   87.839996         -0.000569   
2021-02-05              0.003331  0.000462  ...   87.900002          0.000683   
2021-02-08              0.003827  0.000668  ...   91.470001          0.039811   
...                          ...       ...  ...         ...               ...   
2024-12-24              0.080267  0.000591  ...  126.290001          0.013472   
2024-12-26              0.129918  0.000550  ...  125.059998         -0.009787   
2024-12-27              0.101543  0.000715  ...  125.190002          0.001039   
2024-12-30              0.043333  0.000700  ...  122.440002         -0.022211   
2024-12-31              0.002018  0.000505  ...  120.790001         -0.013568   

                                                             CSCO  \
           Norm_Volume Distance_SMA_20 Close Norm_VWAP      Close   
Date                                                                
2021-02-02   -0.185561             -0.043549 -0.002271  39.865974   
2021-02-03   -0.337739             -0.027521 -0.001136  39.813793   
2021-02-04   -0.366637             -0.036085 -0.001061  41.101185   
2021-02-05   -0.383700             -0.035257 -0.001076  41.823177   
2021-02-08   -0.386411             -0.030421 -0.001037  42.571262   
...                ...                   ...       ...        ...   
2024-12-24    0.224819             -0.060325  0.000079  58.345390   
2024-12-26   -0.358466             -0.040261  0.000049  58.472118   
2024-12-27   -0.348135             -0.045430  0.000048  58.111423   
2024-12-30   -0.144524             -0

In [15]:
close_price_df = df_final.xs('Close', level=1, axis=1)
close_price_df

,AAPL,TSLA,MSFT,NVDA,GOOGL,AMZN,GOOG,META,AVGO,ORCL,CRM,ADBE,AMD,CSCO
Date,,,,,,,,,,,,,,
2021-02-02,131.533722,290.929993,230.249481,13.519486,95.236427,169.000000,95.720467,265.227264,42.982567,58.397964,231.762054,484.929993,88.860001,39.865974
2021-02-03,130.510605,284.896667,233.604568,13.493309,102.172020,165.626495,102.800018,264.800354,41.928913,58.210693,232.375610,481.920013,87.889999,39.813793
2021-02-04,133.872253,283.329987,232.652863,13.626689,101.911491,166.550003,102.417633,264.641357,42.419239,59.315540,235.502716,489.380005,87.839996,41.101185
2021-02-05,133.457504,284.076660,232.835480,13.553641,103.658287,167.607498,104.187027,266.240204,42.002819,59.549629,236.403259,492.119995,87.900002,41.823177
2021-02-08,133.603882,287.806671,233.095016,14.399062,103.444397,166.147003,103.934258,264.730743,42.605808,59.090828,236.442825,493.760010,91.470001,42.571262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,257.286682,462.279999,436.929138,140.189468,195.344940,229.050003,196.932236,605.839600,237.988037,169.721893,342.748352,447.940002,126.290001,58.345390
2024-12-26,258.103729,454.130005,435.715790,139.899521,194.836945,227.050003,196.463745,601.453369,243.627945,169.989227,340.051605,450.160004,125.059998,58.472118
2024-12-27,254.685867,431.660004,428.177216,136.980164,192.007996,223.750000,193.413620,597.924561,240.043442,167.296021,336.797607,446.480011,125.190002,58.111423


In [16]:
target_feature =  "Log_Returns Close"
def create_sequences(df, feature_name, steps: int):
    target_data = df.xs(feature_name, level=1, axis=1)

    sequence_list = []
    # such as steps = 8, loop 0 day to 7 day (include current date)
    for step in range(steps + 1):
        shifted_data = target_data.shift(-step)
        shifted_data.columns = [f"{col}_Day{step}" for col in shifted_data.columns]
        sequence_list.append(shifted_data)

    result_df = pd.concat(sequence_list, axis=1)
    result_df = result_df.sort_index(axis=1)
    return result_df

df_series_seq= create_sequences(df_final, target_feature, steps_sim)
df_series_seq

,AAPL_Day0,AAPL_Day1,AAPL_Day10,AAPL_Day11,AAPL_Day12,AAPL_Day13,AAPL_Day14,AAPL_Day15,AAPL_Day16,AAPL_Day17,...,TSLA_Day82,TSLA_Day83,TSLA_Day84,TSLA_Day85,TSLA_Day86,TSLA_Day87,TSLA_Day88,TSLA_Day89,TSLA_Day9,TSLA_Day90
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-02,0.006317,-0.007809,-0.017802,-0.008674,0.001233,-0.030252,-0.001112,-0.004060,-0.035402,0.002229,...,-0.002114,-0.030563,-0.054820,0.044739,0.010098,-0.002548,-0.008001,0.018761,-0.024686,-0.000377
2021-02-03,-0.007809,0.025432,-0.008674,0.001233,-0.030252,-0.001112,-0.004060,-0.035402,0.002229,0.052452,...,-0.030563,-0.054820,0.044739,0.010098,-0.002548,-0.008001,0.018761,-0.000377,0.002421,0.012708
2021-02-04,0.025432,-0.003103,0.001233,-0.030252,-0.001112,-0.004060,-0.035402,0.002229,0.052452,-0.021115,...,-0.054820,0.044739,0.010098,-0.002548,-0.008001,0.018761,-0.000377,0.012708,-0.013586,-0.030124
2021-02-05,-0.003103,0.001096,-0.030252,-0.001112,-0.004060,-0.035402,0.002229,0.052452,-0.021115,-0.024761,...,0.044739,0.010098,-0.002548,-0.008001,0.018761,-0.000377,0.012708,-0.030124,-0.007752,0.009151
2021-02-08,0.001096,-0.006595,-0.001112,-0.004060,-0.035402,0.002229,0.052452,-0.021115,-0.024761,-0.015938,...,0.010098,-0.002548,-0.008001,0.018761,-0.000377,0.012708,-0.030124,0.009151,-0.089376,0.019207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.011413,0.003171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-26,0.003171,-0.013331,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-27,-0.013331,-0.013352,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
new_columns = []
original_assets = df.columns.levels[0]

for col in df_series_seq.columns:
    for asset in original_assets:
        if col.startswith(asset):
            suffix = col.replace(asset, "") # "_Day+0", "_Day+1"
            new_feature_name = f"{target_feature}{suffix}"
            new_columns.append((asset, new_feature_name))
            break

# df_series_seq
multi_index_cols = pd.MultiIndex.from_tuples(new_columns, names=['Asset', 'Feature'])
df_series_seq.columns = multi_index_cols

df_final_updated = pd.concat([df_final, df_series_seq], axis=1)
df_final_updated = df_final_updated.sort_index(axis=1, level=0)

In [18]:
df_final_updated['AAPL'].columns

Index(['Close', 'Distance_SMA_20 Close', 'Log_Returns Close',
       'Log_Returns Close_Day0', 'Log_Returns Close_Day1',
       'Log_Returns Close_Day10', 'Log_Returns Close_Day11',
       'Log_Returns Close_Day12', 'Log_Returns Close_Day13',
       'Log_Returns Close_Day14', 'Log_Returns Close_Day15',
       'Log_Returns Close_Day16', 'Log_Returns Close_Day17',
       'Log_Returns Close_Day18', 'Log_Returns Close_Day19',
       'Log_Returns Close_Day2', 'Log_Returns Close_Day20',
       'Log_Returns Close_Day21', 'Log_Returns Close_Day22',
       'Log_Returns Close_Day23', 'Log_Returns Close_Day24',
       'Log_Returns Close_Day25', 'Log_Returns Close_Day26',
       'Log_Returns Close_Day27', 'Log_Returns Close_Day28',
       'Log_Returns Close_Day29', 'Log_Returns Close_Day3',
       'Log_Returns Close_Day30', 'Log_Returns Close_Day31',
       'Log_Returns Close_Day32', 'Log_Returns Close_Day33',
       'Log_Returns Close_Day34', 'Log_Returns Close_Day35',
       'Log_Returns Close_D

In [19]:
target_drop_cols = ["Log_Returns Close"]
df_seq_final = df_final_updated.drop(columns=target_drop_cols, level=1).copy()

In [20]:
df_seq_final.dropna(inplace=True)
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])
df_seq_final.shape

Series([], dtype: int64)


(895, 1330)

In [21]:
import re

# 1. แยกกลุ่มคอลัมน์เหมือนเดิม
level_1_cols = df_seq_final.columns.get_level_values(1).unique().tolist()
first = ['Close']
others = [c for c in level_1_cols if c != 'Close' and 'Log_Returns' not in c]

# 2. จัดการกลุ่ม Log_Returns ด้วย Natural Sort
log_rets = [c for c in level_1_cols if 'Log_Returns' in c]

# ฟังก์ชันดึงตัวเลขจากชื่อคอลัมน์ (เช่น 'Day10' -> 10)
def extract_day_number(column_name):
    match = re.search(r'Day(\d+)', column_name)
    return int(match.group(1)) if match else 0

# เรียงลำดับตามตัวเลขที่สกัดได้
log_rets_sorted = sorted(log_rets, key=extract_day_number)

# 3. รวมลำดับใหม่และ reindex
new_level_1_order = first + others + log_rets_sorted
df_seq_final = df_seq_final.reindex(columns=new_level_1_order, level=1)
df_seq_final

AAPL                                              \
                 Close Distance_SMA_20 Close Norm_VWAP Norm_Volume   
Date                                                                 
2021-02-02  131.533722              0.007813  0.000022   -0.094484   
2021-02-03  130.510605              0.011994  0.000484   -0.271330   
2021-02-04  133.872253              0.003154  0.000255   -0.211129   
2021-02-05  133.457504              0.024259  0.000712   -0.237409   
2021-02-08  133.603882              0.019020  0.000691   -0.303623   
...                ...                   ...       ...         ...   
2024-08-16  225.002838              0.029527  0.000285   -0.156716   
2024-08-19  224.843567              0.034794  0.000277   -0.190858   
2024-08-20  225.460693              0.033626  0.000252   -0.252383   
2024-08-21  225.351212              0.035883  0.000191   -0.438286   
2024-08-22  223.489868              0.033623  0.000219   -0.338929   

                                                          \
           Log_Returns Close_Day0 Log_Returns Close_Day1   
Date                                                       
2021-02-02               0.006317              -0.007809   
2021-02-03              -0.007809               0.025432   
2021-02-04               0.025432              -0.003103   
2021-02-05              -0.003103               0.001096   
2021-02-08               0.001096              -0.006595   
...                           ...                    ...   
2024-08-16               0.005901              -0.000708   
2024-08-19              -0.000708               0.002741   
2024-08-20               0.002741              -0.000486   
2024-08-21              -0.000486              -0.008294   
2024-08-22              -0.008294               0.010236   

                                                          \
           Log_Returns Close_Day2 Log_Returns Close_Day3   
Date                                                       
2021-02-02               0.025432              -0.003103   
2021-02-03              -0.003103               0.001096   
2021-02-04               0.001096              -0.006595   
2021-02-05              -0.006595              -0.004569   
2021-02-08              -0.004569              -0.001922   
...                           ...                    ...   
2024-08-16               0.002741              -0.000486   
2024-08-19              -0.000486              -0.008294   
2024-08-20              -0.008294               0.010236   
2024-08-21               0.010236               0.001498   
2024-08-22               0.001498               0.003735   

                                                          ...  \
           Log_Returns Close_Day4 Log_Returns Close_Day5  ...   
Date                                                      ...   
2021-02-02               0.001096              -0.006595  ...   
2021-02-03              -0.006595              -0.004569  ...   
2021-02-04              -0.004569              -0.001922  ...   
2021-02-05              -0.001922               0.001774  ...   
2021-02-08               0.001774              -0.016235  ...   
...                           ...                    ...  ...   
2024-08-16              -0.008294               0.010236  ...   
2024-08-19               0.010236               0.001498  ...   
2024-08-20               0.001498               0.003735  ...   
2024-08-21               0.003735              -0.006776  ...   
2024-08-22              -0.006776               0.014465  ...   

                              TSLA                          \
           Log_Returns Close_Day81 Log_Returns Close_Day82   
Date                                                         
2021-02-02               -0.008965               -0.002114   
2021-02-03               -0.002114               -0.030563   
2021-02-04               -0.030563               -0.054820   
2021-02-05               -0.054820                0.044739   
2021-02-08

In [22]:
df_seq_final.xs("AAPL", level=0, axis=1).columns.unique()

Index(['Close', 'Distance_SMA_20 Close', 'Norm_VWAP', 'Norm_Volume',
       'Log_Returns Close_Day0', 'Log_Returns Close_Day1',
       'Log_Returns Close_Day2', 'Log_Returns Close_Day3',
       'Log_Returns Close_Day4', 'Log_Returns Close_Day5',
       'Log_Returns Close_Day6', 'Log_Returns Close_Day7',
       'Log_Returns Close_Day8', 'Log_Returns Close_Day9',
       'Log_Returns Close_Day10', 'Log_Returns Close_Day11',
       'Log_Returns Close_Day12', 'Log_Returns Close_Day13',
       'Log_Returns Close_Day14', 'Log_Returns Close_Day15',
       'Log_Returns Close_Day16', 'Log_Returns Close_Day17',
       'Log_Returns Close_Day18', 'Log_Returns Close_Day19',
       'Log_Returns Close_Day20', 'Log_Returns Close_Day21',
       'Log_Returns Close_Day22', 'Log_Returns Close_Day23',
       'Log_Returns Close_Day24', 'Log_Returns Close_Day25',
       'Log_Returns Close_Day26', 'Log_Returns Close_Day27',
       'Log_Returns Close_Day28', 'Log_Returns Close_Day29',
       'Log_Returns Close_

In [23]:
# # type(df_final_updated)
# # df
# df_seq_final.xs("AAPL", level=0, axis=1).columns

# level_1_cols = df_seq_final.columns.get_level_values(1).unique().tolist()

# # 2. จัดกลุ่มตามเงื่อนไขเดิม
# first = ['Close']
# log_rets = [c for c in level_1_cols if 'Log_Returns' in c]
# others = [c for c in level_1_cols if c not in first and 'Log_Returns' not in c]

# # รวมลำดับที่ต้องการ: Close -> Others -> Log_Returns
# new_level_1_order = first + others + log_rets

# # 3. ใช้ reindex เพื่อจัดลำดับใหม่ในระดับ Level 1 สำหรับทุกหุ้น
# df_seq_final = df_seq_final.reindex(columns=new_level_1_order, level=1)
# df_seq_final

In [24]:
n_obs = len(df_seq_final)
n_assets = len(df_seq_final.columns.get_level_values(0).unique())
n_features = len(df_seq_final.columns.get_level_values(1).unique())

print(n_obs, n_assets, n_features)

895 14 95


In [25]:
market = df_seq_final.values.reshape(n_obs, n_assets, n_features)
print(type(market))
market.shape

# tensor_list = []
     #    for symbol in self.symbols:
     #        if symbol in self.assets:
     #            # Call Asset method
     #            asset_tensor = self.assets[symbol].to_tensor(features, device)
     #            tensor_list.append(asset_tensor)

     #    # Stack along dimension 1 (Dimension N)
     #    # Asset: [T, F] -> Stack dim=1 -> [T, A, F]
     #    basket_tensor = torch.stack(tensor_list, dim=1)

<class 'numpy.ndarray'>


(895, 14, 95)

In [26]:
df_seq_final.columns.remove_unused_levels()
df_seq_final.columns.get_level_values(1)

Index(['Close', 'Distance_SMA_20 Close', 'Norm_VWAP', 'Norm_Volume',
       'Log_Returns Close_Day0', 'Log_Returns Close_Day1',
       'Log_Returns Close_Day2', 'Log_Returns Close_Day3',
       'Log_Returns Close_Day4', 'Log_Returns Close_Day5',
       ...
       'Log_Returns Close_Day81', 'Log_Returns Close_Day82',
       'Log_Returns Close_Day83', 'Log_Returns Close_Day84',
       'Log_Returns Close_Day85', 'Log_Returns Close_Day86',
       'Log_Returns Close_Day87', 'Log_Returns Close_Day88',
       'Log_Returns Close_Day89', 'Log_Returns Close_Day90'],
      dtype='object', length=1330)

In [27]:
df_seq_final['AAPL'].columns

Index(['Close', 'Distance_SMA_20 Close', 'Norm_VWAP', 'Norm_Volume',
       'Log_Returns Close_Day0', 'Log_Returns Close_Day1',
       'Log_Returns Close_Day2', 'Log_Returns Close_Day3',
       'Log_Returns Close_Day4', 'Log_Returns Close_Day5',
       'Log_Returns Close_Day6', 'Log_Returns Close_Day7',
       'Log_Returns Close_Day8', 'Log_Returns Close_Day9',
       'Log_Returns Close_Day10', 'Log_Returns Close_Day11',
       'Log_Returns Close_Day12', 'Log_Returns Close_Day13',
       'Log_Returns Close_Day14', 'Log_Returns Close_Day15',
       'Log_Returns Close_Day16', 'Log_Returns Close_Day17',
       'Log_Returns Close_Day18', 'Log_Returns Close_Day19',
       'Log_Returns Close_Day20', 'Log_Returns Close_Day21',
       'Log_Returns Close_Day22', 'Log_Returns Close_Day23',
       'Log_Returns Close_Day24', 'Log_Returns Close_Day25',
       'Log_Returns Close_Day26', 'Log_Returns Close_Day27',
       'Log_Returns Close_Day28', 'Log_Returns Close_Day29',
       'Log_Returns Close_

In [28]:
# market[0, 0, 0] # -> that close price


print(f"Market shape:\t\t{market.shape}")
print(f"Close price:\t\t{market[:,:,0:1].shape}")
print(f"Features_Condition:\t{market[:,:,1:n_conditions + 1].shape}")
print(f"Features_Target:\t{market[:,:,n_conditions + 1:].shape}")

print(f"That's a close price:\t{market[0,0,0:1]}")
print(f"Feature_Cond values:\t{market[0, 0, 1:n_conditions + 1]}")
print(f"Feature Target values:\t{market[0, 0, n_conditions + 1:]}")

price = market[:, :, 0:1]
x     = market[:, :, n_conditions + 1:]
cond  = market[:, :, 1:n_conditions + 1]
date  = df_seq_final.index.get_level_values(0).unique().to_numpy()

# print(f"price shape: {price.shape}")
# print(f"x shape: {x.shape}")
# print(f"cond shape: {cond.shape}")
# print(f"date shape: {date.shape}")

Market shape:		(895, 14, 95)
Close price:		(895, 14, 1)
Features_Condition:	(895, 14, 3)
Features_Target:	(895, 14, 91)
That's a close price:	[131.53372192]
Feature_Cond values:	[ 7.81260853e-03  2.22261228e-05 -9.44835414e-02]
Feature Target values:	[ 6.31686751e-03 -7.80877248e-03  2.54315258e-02 -3.10290543e-03
  1.09620923e-03 -6.59523831e-03 -4.56901029e-03 -1.92212124e-03
  1.77428135e-03 -1.62348007e-02 -1.78015863e-02 -8.67390698e-03
  1.23269446e-03 -3.02521914e-02 -1.11166922e-03 -4.06032224e-03
 -3.54019950e-02  2.22911262e-03  5.24515477e-02 -2.11151768e-02
 -2.47606757e-02 -1.59380920e-02  1.06811283e-02 -4.25668569e-02
  3.98453353e-02 -9.20903227e-03  1.63680205e-02 -7.65464817e-03
  2.41624465e-02  1.26624015e-02 -6.47138248e-03 -3.44932661e-02
 -4.49010207e-03  2.79414474e-02 -6.91251035e-03 -2.01957960e-02
  4.15467667e-03  5.12826396e-03  1.48397347e-03 -1.23504890e-02
  1.85916408e-02  6.93444082e-03  2.33039039e-02  2.45905088e-03
  1.33014239e-02  1.90511533e-02  

In [29]:
def calc_ts_splits(total_samples, ratios, gap):
    total_gaps = gap * 2 # Actually there're 3 gap which one was cutted for NaN
    usable_pool = total_samples - total_gaps

    if usable_pool <= 0:
        return "Gap size more than avaliable data"

    train_count = math.floor(usable_pool * ratios[0])
    val_count = math.floor(usable_pool * ratios[1])
    test_count = usable_pool - train_count - val_count

    # -- Train --
    train_start = 0
    train_end = train_count

    # --- Gap 1 ---
    gap1_start = train_end
    gap1_end = gap1_start + gap

    # --- Val ---
    val_start = gap1_end
    val_end = val_start + val_count

    # --- Gap 2 ---
    gap2_start = val_end
    gap2_end = gap2_start + gap

    # --- Test ---
    test_start = gap2_end
    test_end = total_samples

    print(f"--- Configuration: Total {total_samples}, Gap {gap} ---")
    print(f"Train: [{train_start}:{train_end}]\t({train_count} samples)")
    print(f"Gap 1: [{gap1_start}:{gap1_end}]\t(DROPPED)")
    print(f"Val:   [{val_start}:{val_end}]\t({val_count} samples)")
    print(f"Gap 2: [{gap2_start}:{gap2_end}]\t(DROPPED)")
    print(f"Test:  [{test_start}:{test_end}]\t({test_count} samples)")

    return (train_start, train_end), (val_start, val_end), (test_start, test_end)

total_data = len(market)
ratios = [0.8, 0.1, 0.1]
gap = steps_sim

indices   = calc_ts_splits(total_data, ratios, gap)
all_dates = df_seq_final.index.get_level_values(0).unique().to_numpy()
all_prices = market[:, :, 0]

print(f"Dates shape: {all_dates.shape}")
print(f"Prices shape: {all_prices.shape}")
print(f"Market shape: {market.shape}")

--- Configuration: Total 895, Gap 90 ---
Train: [0:572]	(572 samples)
Gap 1: [572:662]	(DROPPED)
Val:   [662:733]	(71 samples)
Gap 2: [733:823]	(DROPPED)
Test:  [823:895]	(72 samples)
Dates shape: (895,)
Prices shape: (895, 14)
Market shape: (895, 14, 95)


In [30]:
(t_s, t_e), (v_s, v_e), (te_s, te_e) = indices

train_part = {
    "x": x[t_s:t_e],
    "cond": cond[t_s:t_e],
    "date": date[t_s:t_e],
    "price": price[t_s:t_e],
}

val_part = {
    "x": x[v_s:v_e],
    "cond": cond[v_s:v_e],
    "date": date[v_s:v_e],
    "price": price[v_s:v_e],
}

test_part = {
    "x": x[te_s:te_e],
    "cond": cond[te_s:te_e],
    "date": date[te_s:te_e],
    "price": price[te_s:te_e],
}

# 3. Print เช็คผล (จะเห็นว่า Total ของ X ทั้ง 3 ก้อน จะไม่เท่ากับ len(market) เพราะหัก Gap ไปแล้ว)
print(f"Train X: {train_part['x'].shape}\nVal X: {val_part['x'].shape}\nTest X: {test_part['x'].shape}")
print("-" * 30)
print(f"Train Cond: {train_part['cond'].shape}\nVal Cond: {val_part['cond'].shape}\nTest Cond: {test_part['cond'].shape}")
print("-" * 30)
print(f"Train Price: {train_part['price'].shape}\nVal Price: {val_part['price'].shape}\nTest Price: {test_part['price'].shape}")
print("-" * 30)
print(f"Train Dates: {train_part['date'].shape}\nVal Dates: {val_part['date'].shape}\nTest Dates: {test_part['date'].shape}")

Train X: (572, 14, 91)
Val X: (71, 14, 91)
Test X: (72, 14, 91)
------------------------------
Train Cond: (572, 14, 3)
Val Cond: (71, 14, 3)
Test Cond: (72, 14, 3)
------------------------------
Train Price: (572, 14, 1)
Val Price: (71, 14, 1)
Test Price: (72, 14, 1)
------------------------------
Train Dates: (572,)
Val Dates: (71,)
Test Dates: (72,)


In [31]:
# scaler_x = StandardScaler()
scaler_cond = StandardScaler()

# Scale X
# x_train_2d = train_part['x'].reshape(train_part['x'].shape[0], -1)
# x_val_2d = val_part['x'].reshape(val_part['x'].shape[0], -1)
# x_test_2d = test_part['x'].reshape(test_part['x'].shape[0], -1)

# scaler_x.fit(x_train_2d)

# train_x_scaled = scaler_x.transform(x_train_2d).reshape(train_part['x'].shape)
# val_x_scaled = scaler_x.transform(x_val_2d).reshape(val_part['x'].shape)
# test_x_scaled = scaler_x.transform(x_test_2d).reshape(test_part['x'].shape)

# inspect(train_x_scaled, "X_scaled - Train Part")
# inspect(val_x_scaled, "X_scaled - Val Part")
# inspect(test_x_scaled, "X_scaled - Test Part")

# Scale Cond
cond_train_2d = train_part['cond'].reshape(train_part['cond'].shape[0], -1)
cond_val_2d = val_part['cond'].reshape(val_part['cond'].shape[0], -1)
cond_test_2d = test_part['cond'].reshape(test_part['cond'].shape[0], -1)

scaler_cond.fit(cond_train_2d)

train_cond_scaled = scaler_cond.transform(cond_train_2d).reshape(train_part['cond'].shape)
val_cond_scaled = scaler_cond.transform(cond_val_2d).reshape(val_part['cond'].shape)
test_cond_scaled = scaler_cond.transform(cond_test_2d).reshape(test_part['cond'].shape)

inspect(train_cond_scaled, "Cond_scaled - Train Part")
inspect(val_cond_scaled, "Cond_scaled - Val Part")
inspect(test_cond_scaled, "Cond_scaled - Test Part")

# Pipeline([('Scaler_X', scaler_x), ('Scaler_Cond', scaler_cond)])

--- Inspecting: Cond_scaled - Train Part ---
------------------------------------
Shape: (572, 14, 3)
Min:   -9.9348
Max:   12.6729
Mean:  0.0000
Std:   1.0000
------------------------------------
--- Inspecting: Cond_scaled - Val Part ---
------------------------------------
Shape: (71, 14, 3)
Min:   -3.3159
Max:   10.7780
Mean:  0.1069
Std:   0.8951
------------------------------------
--- Inspecting: Cond_scaled - Test Part ---
------------------------------------
Shape: (72, 14, 3)
Min:   -4.4689
Max:   15.0113
Mean:  0.1739
Std:   1.0878
------------------------------------


(np.float64(-4.468882635755385),
 np.float64(15.011310053656771),
 np.float64(0.17392874141292058),
 np.float64(1.0877752649737489))

In [32]:

# train_ds = MarketDataset(x=train_x_scaled, cond=train_cond_scaled, date=train_part['date'], price=train_part['price'], window_size=window_size, stride=stride, normalize_window=True)
# val_ds = MarketDataset(x=val_x_scaled, cond=val_cond_scaled, date=val_part['date'], price=val_part['price'], window_size=window_size, stride=stride, normalize_window=True)
# test_ds = MarketDataset(x=test_x_scaled, cond=test_cond_scaled, date=test_part['date'], price=test_part['price'], window_size=window_size, stride=stride, normalize_window=True)


train_ds = MarketDataset(x=train_part['x'], cond=train_cond_scaled, date=train_part['date'], price=train_part['price'], window_size=window_size, stride=stride, normalize_window=True)
val_ds = MarketDataset(x=val_part['x'], cond=val_cond_scaled, date=val_part['date'], price=val_part['price'], window_size=window_size, stride=stride, normalize_window=True)
test_ds = MarketDataset(x=test_part['x'], cond=test_cond_scaled, date=test_part['date'], price=test_part['price'], window_size=window_size, stride=stride, normalize_window=True)

print(f"Num of Windows\nTrain DS: {len(train_ds)}, Val Ds: {len(val_ds)}, Test DS: {len(test_ds)}\n")
print(f"A sample shape from Train DS\n\tx: {train_ds[0]['x'].shape},\n\tcond: {train_ds[0]['cond'].shape}\n\tdates: {len(train_ds[0]['date'])}")

Num of Windows
Train DS: 509, Val Ds: 8, Test DS: 9

A sample shape from Train DS
	x: (91, 64, 14),
	cond: (3, 64, 14)
	dates: 64


In [33]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size_exp, shuffle=False)

batch_train = next(iter(train_loader))
print(len(train_loader))
print(batch_train["x"].shape)
print(batch_train["cond"].shape)
print(batch_train["x"][0,:,0,0])

16
torch.Size([32, 91, 64, 14])
torch.Size([32, 3, 64, 14])
tensor([-0.3263, -0.0902,  0.8089, -2.0522,  0.2258,  0.8723, -0.3028, -1.9954,
        -2.2116, -2.0767,  0.2143,  0.9243, -2.2038,  0.4712,  1.5823, -0.3133,
         1.0517,  1.2413, -0.0547, -1.7200,  0.6471, -0.9813,  0.8262,  0.9434,
         0.4626,  1.1944,  0.2475, -0.7119,  0.3679, -0.0913,  1.0629,  0.5954,
        -0.9607,  1.3044,  0.6963,  0.7858, -0.3428, -0.3297, -0.4156,  1.6272,
         0.2117,  1.6232, -0.2486, -0.3828,  1.7178,  0.0077,  0.0593, -0.0066,
         0.1289,  1.2409, -0.0401,  0.9158,  0.3329,  0.0581,  0.4295,  0.0069,
        -0.5086, -0.8381, -0.0046,  0.1349,  0.6415, -1.4189, -0.4668, -0.5253,
        -0.3681,  0.2140, -0.4869, -0.2814,  0.3972, -0.3093,  0.7574,  1.5058,
        -2.2740,  0.4194, -0.7227, -0.3686,  1.0470,  0.7124, -0.7115, -0.1766,
        -0.5140,  0.1698,  0.3490, -0.3861, -1.8669, -1.1606,  1.2620,  1.0884,
         0.1804, -0.1895, -1.4176])


In [34]:
len(test_loader)

9

In [35]:
next(iter(test_loader))["x"].shape

torch.Size([1, 91, 64, 14])

In [36]:
batch_train["x"][0,:,0,0]
inv_batch_train_x= inverse_transform(batch_train["x"],batch_train["x_mean"], batch_train["x_std"])
inv_batch_train_x[0,:,0,0]

tensor([-0.0054, -0.0009,  0.0167, -0.0393,  0.0052,  0.0174, -0.0051, -0.0366,
        -0.0394, -0.0390,  0.0067,  0.0199, -0.0405,  0.0115,  0.0322, -0.0038,
         0.0213,  0.0242,  0.0000, -0.0302,  0.0129, -0.0182,  0.0160,  0.0188,
         0.0096,  0.0237,  0.0047, -0.0149,  0.0068, -0.0025,  0.0203,  0.0114,
        -0.0209,  0.0264,  0.0134,  0.0150, -0.0081, -0.0074, -0.0089,  0.0337,
         0.0036,  0.0323, -0.0062, -0.0093,  0.0375, -0.0019, -0.0014, -0.0029,
         0.0003,  0.0259, -0.0044,  0.0212,  0.0063, -0.0009,  0.0087, -0.0023,
        -0.0152, -0.0233, -0.0020,  0.0018,  0.0148, -0.0384, -0.0138, -0.0154,
        -0.0107,  0.0047, -0.0137, -0.0082,  0.0092, -0.0097,  0.0187,  0.0378,
        -0.0605,  0.0095, -0.0191, -0.0110,  0.0248,  0.0155, -0.0205, -0.0064,
        -0.0152,  0.0023,  0.0065, -0.0127, -0.0504, -0.0305,  0.0303,  0.0253,
         0.0021, -0.0066, -0.0374])

In [37]:
B, C_cond, T, A = batch_train["cond"].shape
B, C_target, T, A = batch_train["x"].shape
print(f"B: {B}, C_target: {C_target}, C_cond: {C_cond}, T: {T}, A: {A}")

B: 32, C_target: 91, C_cond: 3, T: 64, A: 14


In [38]:
# def objective(trial):
#     d_model = trial.suggest_categorical("d_model", [64, 128, 256])

#     valid_heads = [h for h in [2, 4, 8] if d_model % h == 0]
#     nhead = trial.suggest_categorical("nhead", valid_heads)

#     num_layers = trial.suggest_int("num_layers", 2, 8)
#     dim_feedforward = trial.suggest_int("dim_feedforward", 256, 1024, step=128)

#     # 2. Optimization Params
#     lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
#     weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

#     timesteps = trial.suggest_categorical("timesteps", [500, 1000])
#     beta_start = trial.suggest_float("beta_start", 1e-5, 1e-3, log=True)
#     beta_end = trial.suggest_float("beta_end", 0.01, 0.05)

#     if beta_start >= beta_end:
#         beta_start, beta_end = beta_end, beta_start

#     transformer = DiffusionTransformer(
#         num_assets          = A,
#         num_channels        = C_target,
#         num_cond_channels   = C_cond,
#         num_layers          = num_layers,
#         num_attention_heads = nhead,
#         seq_length          = T,
#         d_model             = d_model,
#         dim_feedforward     = dim_feedforward,
#         dropout             = ddpm_transformer['dropout']
#     ).to(device)

#     diffusion_model = Diffusion(
#         model=transformer,
#         timesteps=timesteps,
#         beta_start=beta_start,
#         beta_end=beta_end
#     ).to(device)

#     optimizer = optim.AdamW(diffusion_model.parameters(), lr=lr, weight_decay=weight_decay)

#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, mode='min', factor=0.5, patience=3
#     )

#     checkpoint_filename = f"tr{trial.number}_d{d_model}_l{num_layers}_h{nhead}_t{ddpm['timesteps']}.pt"


#     engine = Engine(
#         train_loader    = train_loader,
#         val_loader      = val_loader,
#         model           = diffusion_model,
#         optimizer       = optimizer,
#         criterion       = nn.MSELoss(),
#         scheduler       = scheduler,
#         device          = device,
#         checkpoint_dir  = os.path.join(RESULTS_DIR,"optuna_checkpoints"),
#         checkpoint_filename = checkpoint_filename
#     )


#     EPOCHS_PER_TRIAL = 50

#     for epoch in range(1, EPOCHS_PER_TRIAL + 1):
#         try:
#             # Train & Validate
#             train_loss = engine.train(epoch)
#             val_loss = engine.validate(epoch)

#             # --- Key Step: report result to Optuna ---
#             trial.report(val_loss, epoch)

#             # --- Key Step: cut if it's bad (Pruning) ---
#             if trial.should_prune():
#                 print(f"Trial {trial.number} pruned at epoch {epoch}")
#                 raise optuna.exceptions.TrialPruned()

#             # Update Scheduler
#             scheduler.step(val_loss)

#         except RuntimeError as e:
#             # if OOM (Memory is full) skip this Trial  no Crash
#             if "out of memory" in str(e):
#                 torch.cuda.empty_cache()
#                 raise optuna.exceptions.TrialPruned()
#             else:
#                 raise e

#     return val_loss

In [39]:
def objective(trial):
    # ============================================================
    # 1. Diffusion Process (HIGHEST PRIORITY)
    # ============================================================
    # ขยาย Timesteps ให้หลากหลายขึ้น เพราะส่งผลโดยตรงต่อ Quality/Speed trade-off
    timesteps = trial.suggest_categorical("timesteps", [250, 400, 500, 750, 1000])

    # Beta Schedule: ปรับช่วงให้กว้างขึ้นเล็กน้อยเพื่อหา Noise schedule ที่ดีที่สุด
    beta_start = trial.suggest_float("beta_start", 1e-5, 1e-3, log=True)
    beta_end = trial.suggest_float("beta_end", 0.01, 0.03) # บีบช่วงให้แคบลงหน่อยเพื่อความเสถียร

    if beta_start >= beta_end:
        beta_start, beta_end = beta_end, beta_start

    # ============================================================
    # 2. Optimization Params (HIGH PRIORITY)
    # ============================================================
    # LR: ขยาย Upper bound เล็กน้อย (5e-3) เผื่อ model รับไหวและเรียนรู้ได้ไวขึ้น
    # lr = trial.suggest_float("lr", 1e-5, 5e-3, log=True)
    lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)

    # Weight Decay: ขยายช่วงให้กว้างมาก (ถึง 0.1) เพราะ Transformer ต้องการ Regularization ที่เหมาะสม
    # weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-3, 0.05, log=True)

    # ============================================================
    # 3. Architecture Params (MEDIUM - LOW PRIORITY)
    # ============================================================
    # d_model: เพิ่ม 512 เข้ามาสำหรับกรณีที่ต้องการ Model capacity สูง
    d_model = trial.suggest_categorical("d_model", [64, 128, 256, 512])

    # nhead: เพิ่ม 16 heads เพื่อรองรับ d_model ที่ใหญ่ขึ้น (512/16 = 32 dim per head)
    valid_heads = [h for h in [2, 4, 8, 16] if d_model % h == 0]
    nhead = trial.suggest_categorical("nhead", valid_heads)

    # dim_feedforward: ขยาย max ไปถึง 2048 เพื่อรองรับ d_model=512 (ปกติ FFN ~ 4*d_model)
    dim_feedforward = trial.suggest_int("dim_feedforward", 256, 2048, step=128)

    num_layers = trial.suggest_int("num_layers", 2, 8)

    # เพิ่ม Dropout เข้ามาจูนด้วย เพราะสำคัญมากสำหรับ Transformer
    dropout = trial.suggest_float("dropout", 0.1, 0.4, step=0.1)

    # ============================================================
    # Model Setup
    # ============================================================
    transformer = DiffusionTransformer(
        num_assets          = A,
        num_channels        = C_target,
        num_cond_channels   = C_cond,
        num_layers          = num_layers,
        num_attention_heads = nhead,
        seq_length          = T,
        d_model             = d_model,
        dim_feedforward     = dim_feedforward,
        dropout             = dropout  # ใช้ค่าจาก Optuna
    ).to(device)

    diffusion_model = Diffusion(
        model=transformer,
        timesteps=timesteps,
        beta_start=beta_start,
        beta_end=beta_end
    ).to(device)

    optimizer = optim.AdamW(diffusion_model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # ตั้งชื่อไฟล์ให้ละเอียดขึ้น
    checkpoint_filename = f"tr{trial.number}_t{timesteps}_lr{lr:.1e}_dm{d_model}.pt"

    engine = Engine(
        train_loader    = train_loader,
        val_loader      = val_loader,
        model           = diffusion_model,
        optimizer       = optimizer,
        criterion       = nn.MSELoss(),
        scheduler       = scheduler,
        device          = device,
        checkpoint_dir  = checkpoint_dir,
        checkpoint_filename = checkpoint_filename
    )

    # EPOCHS_PER_TRIAL = 50

    for epoch in range(1, EPOCHS_PER_TRIAL + 1):
        try:
            train_loss = engine.train(epoch)
            val_loss = engine.validate(epoch)

            # Reporting
            trial.report(val_loss, epoch)

            # Pruning Check
            if trial.should_prune():
                print(f"✂️ Trial {trial.number} pruned at epoch {epoch} (Loss: {val_loss:.6f})")
                raise optuna.exceptions.TrialPruned()

            scheduler.step(val_loss)

        except RuntimeError as e:
            if "out of memory" in str(e):
                torch.cuda.empty_cache()
                print(f"⚠️ Trial {trial.number} failed due to OOM.")
                raise optuna.exceptions.TrialPruned()
            else:
                raise e

    return val_loss

## Optuna Tuning

In [ ]:
pruner = optuna.pruners.HyperbandPruner(
    min_resource=min_resource,     # เริ่มเช็คตั้งแต่ epoch ที่ 3
    max_resource=max_resource,    # จำนวน epoch สูงสุด
    reduction_factor=reduction_factor
)

study = optuna.create_study(
    study_name="diffusion_tuning_expert",
    direction="minimize",
    pruner=pruner
)

print(f"🚀 Starting Optuna Tuning with {n_trials} trials...")

# ใช้ GC เพื่อเคลียร์แรมระหว่าง Trial
study.optimize(objective, n_trials=n_trials, gc_after_trial=True)

print("\n==================================")
print("✅ Tuning Complete!")
print("==================================")
print(f"🏆 Best Loss: {study.best_value:.6f}")
print("🏆 Best Params:")
for key, value in study.best_params.items():
    print(f"   {key}: {value}")

[I 2026-02-16 00:42:28,835] A new study created in memory with name: diffusion_tuning_expert


🚀 Starting Optuna Tuning with 100 trials...
input dim x: 1274, d_model: 128


2026-02-16 00:42:31,022 - Engine - INFO - Engine initialized on cuda:0
2026-02-16 00:42:31,023 - Engine - INFO - Criterion: MSELoss
Train Ep 1: 100%|██████████| 16/16 [00:02<00:00,  7.14it/s, loss=0.9966]
2026-02-16 00:42:33,301 - Engine - INFO - Epoch 1 | Val Loss: 0.9949
Train Ep 2: 100%|██████████| 16/16 [00:01<00:00,  9.74it/s, loss=0.9935]
2026-02-16 00:42:34,974 - Engine - INFO - Epoch 2 | Val Loss: 0.9908
Train Ep 3: 100%|██████████| 16/16 [00:01<00:00,  9.80it/s, loss=0.9878]
2026-02-16 00:42:36,637 - Engine - INFO - Epoch 3 | Val Loss: 0.9907
Train Ep 4: 100%|██████████| 16/16 [00:01<00:00,  9.78it/s, loss=0.9852]
2026-02-16 00:42:38,306 - Engine - INFO - Epoch 4 | Val Loss: 0.9858
Train Ep 5: 100%|██████████| 16/16 [00:01<00:00,  9.73it/s, loss=0.9810]
2026-02-16 00:42:39,982 - Engine - INFO - Epoch 5 | Val Loss: 0.9802
Train Ep 6: 100%|██████████| 16/16 [00:01<00:00,  9.72it/s, loss=0.9776]
2026-02-16 00:42:41,657 - Engine - INFO - Epoch 6 | Val Loss: 0.9835
Train Ep 7: 100%

input dim x: 1274, d_model: 128


Train Ep 1: 100%|██████████| 16/16 [00:01<00:00,  8.61it/s, loss=0.9957]
2026-02-16 00:44:19,091 - Engine - INFO - Epoch 1 | Val Loss: 0.9931
Train Ep 2: 100%|██████████| 16/16 [00:01<00:00,  8.61it/s, loss=0.9911]
2026-02-16 00:44:20,983 - Engine - INFO - Epoch 2 | Val Loss: 0.9861
Train Ep 3: 100%|██████████| 16/16 [00:01<00:00,  8.60it/s, loss=0.9846]
2026-02-16 00:44:22,883 - Engine - INFO - Epoch 3 | Val Loss: 0.9863
Train Ep 4: 100%|██████████| 16/16 [00:01<00:00,  8.61it/s, loss=0.9801]
2026-02-16 00:44:24,776 - Engine - INFO - Epoch 4 | Val Loss: 0.9835
Train Ep 5: 100%|██████████| 16/16 [00:01<00:00,  8.65it/s, loss=0.9786]
2026-02-16 00:44:26,664 - Engine - INFO - Epoch 5 | Val Loss: 0.9762
Train Ep 6: 100%|██████████| 16/16 [00:01<00:00,  8.48it/s, loss=0.9723]
2026-02-16 00:44:28,590 - Engine - INFO - Epoch 6 | Val Loss: 0.9699
Train Ep 7: 100%|██████████| 16/16 [00:01<00:00,  8.58it/s, loss=0.9655]
2026-02-16 00:44:30,492 - Engine - INFO - Epoch 7 | Val Loss: 0.9658
Train 

...

2026-02-16 00:56:21,260 - Engine - INFO - Engine initialized on cuda:0
2026-02-16 00:56:21,261 - Engine - INFO - Criterion: MSELoss


input dim x: 1274, d_model: 512


Train Ep 1: 100%|██████████| 16/16 [00:02<00:00,  6.28it/s, loss=0.9892]
2026-02-16 00:56:23,857 - Engine - INFO - Epoch 1 | Val Loss: 0.9883
Train Ep 2: 100%|██████████| 16/16 [00:02<00:00,  6.35it/s, loss=0.9692]
2026-02-16 00:56:26,428 - Engine - INFO - Epoch 2 | Val Loss: 0.9636
Train Ep 3: 100%|██████████| 16/16 [00:02<00:00,  6.48it/s, loss=0.9456]
2026-02-16 00:56:28,943 - Engine - INFO - Epoch 3 | Val Loss: 0.9605
Train Ep 4: 100%|██████████| 16/16 [00:02<00:00,  6.55it/s, loss=0.9313]
2026-02-16 00:56:31,430 - Engine - INFO - Epoch 4 | Val Loss: 0.9201
Train Ep 5: 100%|██████████| 16/16 [00:02<00:00,  6.55it/s, loss=0.9211]
2026-02-16 00:56:33,917 - Engine - INFO - Epoch 5 | Val Loss: 0.8981
Train Ep 6: 100%|██████████| 16/16 [00:02<00:00,  6.55it/s, loss=0.8988]
2026-02-16 00:56:36,404 - Engine - INFO - Epoch 6 | Val Loss: 0.8661
Train Ep 7: 100%|██████████| 16/16 [00:02<00:00,  6.56it/s, loss=0.8808]
2026-02-16 00:56:38,886 - Engine - INFO - Epoch 7 | Val Loss: 0.9026
Train 

input dim x: 1274, d_model: 512


Train Ep 1: 100%|██████████| 16/16 [00:02<00:00,  6.29it/s, loss=0.9932]
2026-02-16 00:58:22,401 - Engine - INFO - Epoch 1 | Val Loss: 0.9906
Train Ep 2: 100%|██████████| 16/16 [00:01<00:00,  8.27it/s, loss=0.9746]
2026-02-16 00:58:24,378 - Engine - INFO - Epoch 2 | Val Loss: 0.9623
Train Ep 3: 100%|██████████| 16/16 [00:01<00:00,  8.29it/s, loss=0.9599]
2026-02-16 00:58:26,347 - Engine - INFO - Epoch 3 | Val Loss: 0.9414
Train Ep 4: 100%|██████████| 16/16 [00:01<00:00,  8.30it/s, loss=0.9359]
2026-02-16 00:58:28,312 - Engine - INFO - Epoch 4 | Val Loss: 0.9473
Train Ep 5: 100%|██████████| 16/16 [00:01<00:00,  8.48it/s, loss=0.9209]
2026-02-16 00:58:30,234 - Engine - INFO - Epoch 5 | Val Loss: 0.9183
Train Ep 6: 100%|██████████| 16/16 [00:01<00:00,  8.48it/s, loss=0.9094]
2026-02-16 00:58:32,155 - Engine - INFO - Epoch 6 | Val Loss: 0.9298
Train Ep 7: 100%|██████████| 16/16 [00:01<00:00,  8.42it/s, loss=0.8891]
2026-02-16 00:58:34,090 - Engine - INFO - Epoch 7 | Val Loss: 0.9096
Train 

In [ ]:
# n_trails = 50
# n_warmup = 10

# study = optuna.create_study(
#         study_name="diffusion_tuning",
#         direction="minimize",  # เราต้องการ Loss ต่ำสุด
#         pruner=optuna.pruners.MedianPruner(n_warmup_steps=n_warmup) # ให้โอกาส 3 Epoch แรกก่อนค่อยตัด
#     )

# print("🚀 Starting Optuna Tuning...")


# study.optimize(objective, n_trials=n_trails)

# print("\n==================================")
# print("✅ Tuning Complete!")
# print("==================================")
# print(f"Best Loss: {study.best_value:.6f}")
# print("Best Params:")
# for key, value in study.best_params.items():
#     print(f"  {key}: {value}")

# best_params = study.best_params

## Optuna Params

In [ ]:
from optuna.visualization import plot_param_importances, plot_optimization_history

# 1. ดูว่า Param ไหนส่งผลต่อ Loss มากสุด
fig1 = plot_param_importances(study)
fig1.show(renderer="notebook")

# 2. ดูแนวโน้มว่า Loss ลดลงเรื่อยๆ ไหม
fig2 = plot_optimization_history(study)
fig2.show(renderer="notebook")

In [ ]:
study

In [ ]:
import optuna.visualization as vis

fig1 = vis.plot_optimization_history(study)
fig1.show(renderer="notebook")

In [ ]:
best_params = study.best_params
best_params

In [ ]:
print(f"lr: {lr}, weight_decay: {weight_decay},  ddpm_transformer['dropout']: { ddpm_transformer['dropout']}")

## Fit Model with Best Params from Optuna

In [ ]:
# ignore droupout
# lr
# too high

# Optimizer
weight_decay = best_params['weight_decay']
lr = best_params['lr']

model = DiffusionTransformer(
    num_assets          = A,
    num_channels        = C_target,
    num_cond_channels   = C_cond,
    num_layers          = best_params['num_layers'],
    num_attention_heads = best_params['nhead'],
    seq_length          = T,
    d_model             = best_params['d_model'],
    dim_feedforward     = best_params['dim_feedforward'],
    dropout             = ddpm_transformer['dropout']
).to(device)

# diffusion = Diffusion(
#     model=model,
#     timesteps=ddpm['timesteps'],
#     beta_start=ddpm['beta_start'],
#     beta_end=ddpm['beta_end']
# ).to(device)

diffusion = Diffusion(
    model=model,
    timesteps=best_params['timesteps'],
    beta_start=best_params['beta_start'],
    beta_end=best_params['beta_end']
).to(device)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, cooldown=2, threshold=0.01)

engine = Engine(
    train_loader    = train_loader,
    val_loader      = val_loader,
    model           = diffusion,
    optimizer       = optimizer,
    criterion       = nn.MSELoss(),
    scheduler       = scheduler,
    device          = device,
    checkpoint_dir  = checkpoint_dir,
    checkpoint_filename = checkpoint_filename
)

checkpoint_dir = os.path.join(RESULTS_DIR,"experimental_checkpoints_no_norm")
checkpoint_filename = f"tr{n_trials}_d{ddpm_transformer['d_model']}_dff{ddpm_transformer['dim_feedforward']}_l{ddpm_transformer['num_layers']}_h{ddpm_transformer['nhead']}_t{ddpm['timesteps']}.pt"

In [ ]:
engine.fit(epochs)

# Experimental

In [ ]:
batch_test = next(iter(test_loader))
print(f"x: {batch_test['x'].shape}, x_cond: {batch_test['cond'].shape}, dates: {batch_test['date'].shape}, prices: {batch_test['price'].shape}")

In [ ]:
def transform_dates(dates: torch.Tensor) -> pd.DatetimeIndex:
    dates_np = dates.cpu().numpy()
    dates_pd = pd.to_datetime(dates_np.flatten(), unit='ns')
    return dates_pd

In [ ]:
engine.simulate('')

In [ ]:
engine.plot_history()

In [ ]:
12

In [ ]:
def plot_final_revised(real_data, mc_genai, mc_stat, asset_idx=0, context_window=2):
    # --- 1. Data Preparation (Anchoring Logic เหมือนเดิม) ---
    real_series = real_data[:, asset_idx].numpy()
    genai_series = mc_genai[:, :, asset_idx].numpy().T
    stat_series = mc_stat[:, :, asset_idx].numpy().T

    total_len = len(real_series)
    sim_len = genai_series.shape[0]
    anchor_idx = total_len - sim_len - 1

    # เชื่อมจุด
    anchor_value = real_series[anchor_idx]
    anchor_row = np.full((1, genai_series.shape[1]), anchor_value)

    genai_plot = np.vstack([anchor_row, genai_series])
    stat_plot = np.vstack([anchor_row, stat_series])

    time_sim_extended = np.arange(anchor_idx, total_len)
    time_all = np.arange(total_len)
    start_plot_idx = max(0, anchor_idx - context_window + 1)

    # --- 2. Color & Style Setup ---
    plt.figure(figsize=(12, 6), dpi=120)

    # === COLORS PALETTE ===
    c_real = '#06C755'    # Vivid Green (ค่าเดิมที่คุณชอบ)

    # สีใหม่: ใช้โทนที่สว่างและ Clean ขึ้น (Material Colors)
    c_genai = '#FF5252'   # Red Accent (แดงสว่าง อมชมพูนิดๆ ดูไม่เก่า)
    c_stat = '#448AFF'    # Blue Accent (น้ำเงินฟ้า ดูเป็น Professional Stat)

    # สีเส้น Mean (ปรับให้เข้มกว่าตัวเส้นนิดหน่อย)
    c_genai_mean = '#D32F2F'
    c_stat_mean = '#1976D2'

    # --- 3. Plotting Simulations (Thicker Lines) ---
    # linewidth=1.5 (หนาขึ้นอีก ตามขอ)
    # alpha=0.06 (ปรับลดลงนิดนึงเพราะเส้นหนาขึ้น เดี๋ยวทับกันแล้วทึบเกิน)

    # Stat Sim (Blue)
    plt.plot(time_sim_extended, stat_plot, color=c_stat,
             linewidth=1.5, alpha=0.06, zorder=1)

    # GenAI Sim (Red)
    plt.plot(time_sim_extended, genai_plot, color=c_genai,
             linewidth=1.5, alpha=0.06, zorder=2)

    # --- 4. Mean Lines (เส้นนำสายตา) ---
    plt.plot(time_sim_extended, np.mean(stat_plot, axis=1), color=c_stat_mean,
             linewidth=2, linestyle='--', zorder=3, alpha=0.9)

    plt.plot(time_sim_extended, np.mean(genai_plot, axis=1), color=c_genai_mean,
             linewidth=2, linestyle='--', zorder=4, alpha=0.9)

    # --- 5. Real Data (The Hero) ---
    # Halo Effect (ขอบขาว)
    plt.plot(time_all[start_plot_idx:], real_series[start_plot_idx:],
             color='white', linewidth=4.5, zorder=5) # เพิ่มขอบขาวให้หนารับกับเส้นเขียว

    # Real Line
    plt.plot(time_all[start_plot_idx:], real_series[start_plot_idx:],
             color=c_real, linewidth=2.5, marker='o', markersize=6, zorder=6)

    # --- 6. Decoration ---
    plt.axvline(x=anchor_idx, color='gray', linestyle=':', linewidth=1.5, alpha=0.4)

    plt.title(f'Asset {asset_idx} Forecast Analysis', fontsize=14, fontweight='bold', color='#333333', pad=15)
    plt.xlabel('Time Step', fontsize=11)
    plt.ylabel('Log Returns', fontsize=11)
    plt.xlim(start_plot_idx, total_len - 0.5)

    # Grid บางๆ สบายตา
    plt.grid(True, alpha=0.15, linestyle='--')

    # Legend
    custom_lines = [
        Line2D([0], [0], color=c_real, lw=2.5, marker='o', label='Ground Truth'),
        Line2D([0], [0], color=c_genai, lw=2, label='GenAI Model'),
        Line2D([0], [0], color=c_stat, lw=2, label='Statistical Model')
    ]
    plt.legend(handles=custom_lines, loc='upper left', frameon=True,
               facecolor='white', framealpha=0.95, edgecolor='#E0E0E0')

    plt.tight_layout()
    plt.show()

In [ ]:
def expand_mc_size(x: torch.Tensor, cond: torch.Tensor, n_samples: int):
    return x.expand(n_samples, -1, -1, -1), cond.expand(n_samples, -1, -1, -1)

def transform_dates(dates: torch.Tensor, extend_days: int = 0) -> pd.DatetimeIndex:
    dates_np = dates.cpu().numpy()
    dates_pd = pd.to_datetime(dates_np.flatten(), unit='ns')


    extend_days: int = 0
    if extend_days > 0:
        last_date = dates_pd[-1]
        extended_range = pd.date_range(
            start=last_date + pd.Timedelta(days=1),
            periods=extend_days,
            freq='D'
        )
        dates_pd = dates_pd.append(extended_range)

    return dates_pd

n_samples = 1000


def evaluation(dataloader, num_samples_sim: int, rebalance_days: int = 1):
    count = 0

    eval_dir = os.path.join(REPORTS_QS_DIR, f"eval_{checkpoint_filename[:-3]}")
    os.makedirs(eval_dir, exist_ok=True)

    for batch in dataloader:
        # Monte Carlo
        # MC Genai
        x_mc, cond_mc = expand_mc_size(batch['x'], batch['cond'], num_samples_sim)
        print(f"x_mc: {x_mc.shape} , cond_mc: {cond_mc.shape}")

        # simulate with inpaiting
        sim_full_scaled, sim_only_scaled =  engine.simulate(x_mc.to(device), cond_mc.to(device))

        sim_only = inverse_transform(sim_full_scaled.to(device), batch['x_mean'].to(device), batch['x_std'].to(device))
        print(f"sim_only: {sim_only.shape}")


        mc_sim_genai = sim_only[:, 1:, -1:, :].squeeze(2)
        print(f"mc_sim_genai: {mc_sim_genai.shape}")

        steps_sim = mc_sim_genai.shape[1]
        print(f"steps_sim: {steps_sim}")

        # MC Stats
        x_unscaled = inverse_transform(batch['x'], batch['x_mean'], batch['x_std'])
        close_log_returns = x_unscaled[0, 0, :, :]
        print(f"Close log returns: {close_log_returns.shape}")

        mc_sim_stat = monte_carlo_statistic(torch.as_tensor(close_log_returns), n_sims=num_samples_sim, steps=steps_sim) # [n_sims, steps, assets]
        print(f"mc_sim_stat: {mc_sim_stat.shape}")

        # Ground truth
        gt = x_unscaled
        gt = gt[0, 1:, -1:, :]
        gt = gt.squeeze(1)
        print(f"gt: {gt.shape}")

        # Dates
        dates = transform_dates(batch['date'], steps_sim)[-steps_sim:]
        print(f"dates: {dates.shape}")

        # Portfolio management
        B_genai, F_genai, A_genai = mc_sim_genai.shape
        sim_genai_dist = mc_sim_genai.reshape(-1, A_genai)

        B_stat, F_stat, A_stat = mc_sim_stat.shape
        sim_stat_dist = mc_sim_stat.reshape(-1, A_stat)

        risk_free_rate = 0.02/252
        weight_bounds = (0, 1)
        is_calc_distribution = True

        # New Dir per Batch contain sim
        # bt means batch test
        report_dir = os.path.join(eval_dir, f"bt{count}")
        os.makedirs(report_dir, exist_ok=True)

        for sim_idx in range(B_genai):
            sim_genai = mc_sim_genai[sim_idx, :, :]
            sim_stat  = mc_sim_stat[sim_idx, :, :]
            print(sim_genai.shape, sim_stat.shape)

            report_filename = f"sim{sim_idx}"
            portfolio = Portfolio(risk_free_rate, weight_bounds=weight_bounds, save_dir=report_dir)


            mu_genai, sigma_genai = portfolio.calc(sim_genai.cpu().numpy())
            mu_stats, sigma_stats = portfolio.calc(sim_stat.cpu().numpy())

            print(f"Mu_genai: {mu_genai.shape}, Sigma_genai: {sigma_genai.shape}")
            print(f"Mu_stats: {mu_stats.shape}, Sigma_stats: {sigma_stats.shape}")
            # logger.debug(f"Mu_genai: {mu_genai.shape}, Sigma_genai: {sigma_genai.shape}")
            # logger.debug(f"Mu_stats: {mu_stats.shape}, Sigma_stats: {sigma_stats.shape}")

            weights_genai = portfolio.optimize_weights(mu_genai, sigma_genai, risk_free_rate=risk_free_rate, scipy=True)
            weights_stats = portfolio.optimize_weights(mu_stats, sigma_stats, risk_free_rate=risk_free_rate, scipy=True)

            print(f"weights_genai: {weights_genai.shape}")
            print(f"weights_stats: {weights_stats.shape}")

            # logger.debug(f"weights_genai: {weights_genai.shape}")
            # logger.debug(f"weights_stats: {weights_stats.shape}")

            report = portfolio.back_test(weights=weights_genai, weights_benchmark=weights_stats, returns=gt.cpu().numpy(), dates=dates, is_saved=True, filename=report_filename, rebalance_days=rebalance_days)

        logs_filename = os.path.join(report_dir, f"logs_data.pt")
        torch.save({
            "x_unscaled": x_unscaled,
            "dates": dates,
            "mc_sim_genai": mc_sim_genai,
            "mc_sim_stat": mc_sim_stat,
            "gt": gt
        }, logs_filename)

        # plot_final_revised(real_data, mc_genai, mc_stat, asset_idx=0, context_window=2)

        count += 1
        break

In [ ]:
evaluation(test_loader, 1000)

In [ ]:
fig1 = plot_param_importances(study)
fig1.show(renderer="notebook")

# 2. ดูแนวโน้มว่า Loss ลดลงเรื่อยๆ ไหม
fig2 = plot_optimization_history(study)
fig2.show(renderer="notebook")

In [ ]:
best_params = study.best_params
best_params

# Experimental

In [ ]:
def extract_simulation_stats(sim_data: np.ndarray, method: str = 'separate'):
    """
    สกัดค่า Mu และ Sigma จากข้อมูล Simulation หลายเส้นทาง

    Parameters:
    - sim_data: numpy array รูปแบบ [N_sims, Steps, Assets] (เช่น 1000, 20, 5)
    - method: 'combined' (เทรวมกัน) หรือ 'separate' (คิดแยกทีละเส้นทางแล้วเฉลี่ย)

    Returns:
    - mu_final: numpy array รูปแบบ [Assets,]
    - sigma_final: numpy array รูปแบบ [Assets, Assets]
    """
    N_sims, Steps, Assets = sim_data.shape

    if method == 'combined':
        # วิธีที่ 1: รวมทุกเส้นทางเป็น Distribution เดียว
        flattened_data = sim_data.reshape(-1, Assets)
        mu_final = calc_expected_returns(flattened_data)
        sigma_final = calc_covariance(flattened_data)

    elif method == 'separate':
        # วิธีที่ 2: คิดแยกทีละเส้นทาง (Path-by-Path)
        mu_list = []
        sigma_list = []

        for i in range(N_sims):
            path_data = sim_data[i, :, :] # ดึงมาทีละ 1 เส้นทาง

            # เรียกใช้ฟังก์ชันย่อย
            mu_path = calc_expected_returns(path_data)
            sigma_path = calc_covariance(path_data)

            mu_list.append(mu_path)
            sigma_list.append(sigma_path)

        # นำค่าสถิติของทั้ง 1,000 เส้นทางมาหาค่าเฉลี่ย
        mu_final = np.mean(mu_list, axis=0)
        sigma_final = np.mean(sigma_list, axis=0)

    else:
        raise ValueError("Parameter 'method' ต้องเป็น 'combined' หรือ 'separate' เท่านั้น")

    return mu_final, sigma_final

In [ ]:
from scipy.stats import skew, kurtosis

def get_generation_level_stats(data: np.ndarray, model_name: str, is_gt: bool = False) -> pd.DataFrame:
    """
    ดึงค่าสถิติ Model Performance ระดับ Generation Level

    Parameters:
    - data: หากเป็น GenAI/Stat จะเป็น shape [N_paths, Steps, Assets]
            หากเป็น Ground Truth (GT) จะเป็น shape [Steps, Assets]
    - model_name: ชื่อโมเดล เช่น 'GenAI', 'GBM-Stat', 'Ground-Truth'
    - is_gt: Flag ระบุว่าเป็น Ground Truth หรือไม่

    Returns:
    - pd.DataFrame สรุปค่าสถิติของแต่ละ Asset
    """
    # ถ้าเป็นข้อมูลจำลอง (มี N_paths) เราจะรวบทุกเส้นทางมาดู Distribution รวม
    if not is_gt and len(data.shape) == 3:
        N_paths, Steps, Assets = data.shape
        data_flat = data.reshape(-1, Assets) # รวมเป็น [N_paths * Steps, Assets]
    else:
        # ถ้าเป็น GT จะมีแค่ 2 มิติ [Steps, Assets] อยู่แล้ว
        Assets = data.shape[-1]
        data_flat = data

    stats_list = []

    # คำนวณสถิติทีละ Asset
    for asset_idx in range(Assets):
        asset_data = data_flat[:, asset_idx]

        stats = {
            'Model': model_name,
            'Asset_Idx': asset_idx,
            'Mean': np.mean(asset_data),
            'Std (Volatility)': np.std(asset_data),
            'Min': np.min(asset_data),
            'Max': np.max(asset_data),
            'Skewness': skew(asset_data),      # ความเบ้ (ดูว่าผลตอบแทนเอียงไปทางบวกหรือลบ)
            'Kurtosis': kurtosis(asset_data)   # ความโด่ง (ดูความเสี่ยงของหาง (Fat-tail risk))
        }
        stats_list.append(stats)

    return pd.DataFrame(stats_list)

In [ ]:
def evaluation(dataloader, engine, num_samples_sim: int = 1000, is_dynamic: bool = False, rebalance_freq: int = 5):
    """
    is_dynamic: หากเป็น False คือ Static (จัดครั้งเดียว) | หากเป็น True คือจัดพอร์ตแบบ Rolling
    rebalance_freq: จำนวนวันที่จะถือพอร์ตก่อนทำการจำลองและ Optimize ใหม่ (ใช้เมื่อ is_dynamic=True)
    """
    eval_dir = os.path.join(REPORTS_QS_DIR, f"eval_{checkpoint_filename[:-3]}")
    os.makedirs(eval_dir, exist_ok=True)

    count = 0

    for batch in dataloader:
        print(f"\n{'='*40}\n🚀 ประมวลผล Batch ที่ {count}\n{'='*40}")

        # -----------------------------------------------------
        # 1. GENERATION LEVEL: จำลองข้อมูล
        # -----------------------------------------------------
        # GenAI Simulation
        x_mc, cond_mc = expand_mc_size(batch['x'], batch['cond'], num_samples_sim)
        sim_full_scaled, sim_only_scaled = engine.simulate(x_mc.to(device), cond_mc.to(device))
        sim_only = inverse_transform(sim_full_scaled.to(device), batch['x_mean'].to(device), batch['x_std'].to(device))
        mc_sim_genai = sim_only[:, 1:, -1:, :].squeeze(2) # [1000, Steps, Assets]

        # Stat Simulation (GBM)
        x_unscaled = inverse_transform(batch['x'], batch['x_mean'], batch['x_std'])
        close_log_returns = x_unscaled[0, 0, :, :]
        steps_sim = mc_sim_genai.shape[1]
        mc_sim_stat = monte_carlo_statistic(torch.as_tensor(close_log_returns), n_sims=num_samples_sim, steps=steps_sim)

        # Ground Truth
        gt = x_unscaled[0, 1:, -1:, :].squeeze(1) # [Steps, Assets]
        dates = transform_dates(batch['date'], steps_sim)[-steps_sim:]

        # -----------------------------------------------------
        # 2. MODEL PERFORMANCE: ดึงตารางสถิติ (ตาม Workflow ของคุณ)
        # -----------------------------------------------------
        sim_genai_np = mc_sim_genai.detach().cpu().numpy()
        sim_stat_np = mc_sim_stat.detach().cpu().numpy()
        gt_np = gt.detach().cpu().numpy()

        df_stats_genai = get_generation_level_stats(sim_genai_np, model_name="GenAI")
        df_stats_stat  = get_generation_level_stats(sim_stat_np, model_name="GBM-Stat")
        df_stats_gt    = get_generation_level_stats(gt_np, model_name="GroundTruth", is_gt=True)

        # รวมตารางสถิติเพื่อดูเปรียบเทียบ
        df_performance = pd.concat([df_stats_gt, df_stats_stat, df_stats_genai], ignore_index=True)
        print("\n📊 Model Performance (Generation Level):")
        print(df_performance.to_string())
        # สามารถเซฟ df_performance ลง .csv ได้ตรงนี้

        # -----------------------------------------------------
        # 3. PORTFOLIO OPTIMIZATION & BACKTEST
        # -----------------------------------------------------
        risk_free_rate = 0.02 / 252
        weight_bounds = (0, 1)
        report_dir = os.path.join(eval_dir, f"bt{count}")
        os.makedirs(report_dir, exist_ok=True)
        portfolio = Portfolio(risk_free_rate, weight_bounds=weight_bounds, save_dir=report_dir)

        if not is_dynamic:
            # === แบบที่ 1: STATIC PORTFOLIO (ทำเหมือนเดิมที่คุณมี) ===
            print("\n📈 โหมด: Static Portfolio (Buy & Hold)")
            # 1. สกัดค่า Mu, Sigma รวม
            mu_genai, sigma_genai = extract_simulation_stats(sim_genai_np, method='separate')
            mu_stats, sigma_stats = extract_simulation_stats(sim_stat_np, method='separate')

            # 2. Optimize Weights ครั้งเดียวสำหรับหน้าต่างเวลานี้
            weights_genai = portfolio.optimize_weights(mu_genai, sigma_genai, risk_free_rate=risk_free_rate, scipy=True)
            weights_stats = portfolio.optimize_weights(mu_stats, sigma_stats, risk_free_rate=risk_free_rate, scipy=True)

            # 3. Backtest ผลลัพธ์รวดเดียว
            report_filename = f"static_comparison_report"
            report = portfolio.back_test(
                weights=weights_genai, weights_benchmark=weights_stats,
                returns=gt_np, dates=dates, is_saved=True,
                filename=report_filename, rebalance_days=1 # 1 คือคิดผลตอบแทนรายวัน แต่ไม่ได้เปลี่ยนน้ำหนัก
            )

        else:
            # === แบบที่ 2: DYNAMIC PORTFOLIO (โครงสร้างการทำ Rolling Rebalance) ===
            print(f"\n🔄 โหมด: Dynamic Portfolio (Rebalance ทุกๆ {rebalance_freq} วัน)")
            """
            แนวคิดสำหรับการทำ Dynamic:
            ในโลกความจริงเราจะไม่รู้ GT ทั้งเส้น เราจึงต้องซอย Loop ย่อยตาม rebalance_freq
            """
            portfolio_returns_genai = []
            portfolio_returns_stats = []

            # สมมติว่า steps_sim = 20, rebalance_freq = 5
            for start_idx in range(0, steps_sim, rebalance_freq):
                end_idx = min(start_idx + rebalance_freq, steps_sim)

                # ข้อควรระวัง: ในการทำ Dynamic ที่แท้จริง ตรงนี้คุณต้องดึง "บริบทของตลาด ณ วันที่ start_idx"
                # มาเข้าโมเดล GenAI/Stat ใหม่ เพื่อจำลองอนาคตระยะสั้นของลูปนี้
                # (แต่ในโค้ดนี้เราดึงจากก้อนใหญ่ที่จำลองไว้แล้วมาหั่นเป็นช่วงๆ เพื่อแสดงแนวคิดก่อน)

                current_sim_genai = sim_genai_np[:, start_idx:end_idx, :]
                current_sim_stat  = sim_stat_np[:, start_idx:end_idx, :]
                current_gt        = gt_np[start_idx:end_idx, :]

                # 1. ดึงสถิติของช่วงเวลานี้
                mu_g, cov_g = extract_simulation_stats(current_sim_genai, method='separate')
                mu_s, cov_s = extract_simulation_stats(current_sim_stat, method='separate')

                # 2. หา Weights ใหม่ (Rebalance)
                w_g = portfolio.optimize_weights(mu_g, cov_g, risk_free_rate=risk_free_rate, scipy=True)
                w_s = portfolio.optimize_weights(mu_s, cov_s, risk_free_rate=risk_free_rate, scipy=True)

                # 3. คำนวณผลตอบแทนของพอร์ตในช่วงเวลานี้ (นำน้ำหนักไปคูณกับผลตอบแทนจริง GT)
                # w_g shape: (Assets,) | current_gt shape: (Freq, Assets) -> ผลลัพธ์: (Freq,)
                port_ret_g = np.sum(current_gt * w_g, axis=1)
                port_ret_s = np.sum(current_gt * w_s, axis=1)

                portfolio_returns_genai.extend(port_ret_g)
                portfolio_returns_stats.extend(port_ret_s)

            print("✅ จำลองการปรับพอร์ตแบบ Dynamic เสร็จสิ้น (กรุณาใช้ QuantStats เพื่อวิเคราะห์ `portfolio_returns_genai` ต่อ)")
            # สามารถต่อยอดนำ portfolio_returns_genai แปลงเป็น pd.Series แล้วป้อนเข้า QuantStats ได้เลย

        count += 1
        break # Test 1 batch ก่อน

In [ ]:
evaluation(test_loader, engine, is_dynamic=True, report_name="test_run_v1")